# 27c — Catching Negative Candidate Technical Validation

## Objetivo

Validar técnicamente los repositorios candidatos asociados a sitios de Catching que ya tienen al menos una etiqueta Nielsen negativa.

Este notebook aparece porque `27b` confirmó dos hechos:

1. no existe todavía ningún sitio negativo con `CONFIRMED_STRONG`;
2. sí existen repositorios candidatos reales, algunos con evidencia `MODERATE` y otros aún `UNKNOWN`.

La meta no es convertir un candidato en ground truth por conveniencia. La meta es separar, con criterios reproducibles, cuatro niveles de evidencia de fuente:

```text
A_CONFIRMED_STRONG
B_MODERATE_VALIDATED
C_EXACT_NAME_TECHNICAL
D_ACCESSIBLE_CANDIDATE
E_NOT_RECOVERABLE
```

Solo A es equivalente al nivel estricto usado previamente.

B y C podrán usarse, si procede, en un **seed etiquetado extendido/weak-source-match**, siempre conservando esa procedencia. Nunca se mezclarán silenciosamente con A.

Este notebook:

- verifica que el repositorio exista;
- hace clone superficial no destructivo;
- verifica que tenga código fuente web;
- recupera evidencia de validaciones anteriores;
- comprueba coincidencia exacta entre nombre del sitio y nombre del repositorio;
- prioriza los negativos necesarios para H1/H4/H5/H6;
- decide si puede avanzarse a extracción de X.

No extrae todavía las tres X.


## 1. Configuración

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import subprocess
import re
import shutil
import os
import json

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_colwidth", 500)

PROJECT_DIR = Path("..").resolve()

REPORTS_DIR = PROJECT_DIR / "reports" / "tables"
PUBLIC_INTERIM_DIR = PROJECT_DIR / "data" / "public_v2" / "interim"

SEMISUP_REPORTS_DIR = REPORTS_DIR / "semisupervised"
RECOVERY_ROOT = (
    PROJECT_DIR
    / "data"
    / "public_v2"
    / "source_recovery"
    / "catching_negative_candidates"
)

SEMISUP_REPORTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RECOVERY_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

TARGETS = [
    "H1_violation",
    "H4_violation",
    "H5_violation",
    "H6_violation"
]

WEB_EXTENSIONS = {
    ".html",
    ".htm",
    ".css",
    ".scss",
    ".sass",
    ".less",
    ".js",
    ".jsx",
    ".ts",
    ".tsx",
    ".vue",
    ".svelte"
}

print("PROJECT_DIR:", PROJECT_DIR)
print("RECOVERY_ROOT:", RECOVERY_ROOT)


PROJECT_DIR: /Users/uriel/Documents/machine_learning
RECOVERY_ROOT: /Users/uriel/Documents/machine_learning/data/public_v2/source_recovery/catching_negative_candidates


## 2. Cargar registro de recuperación del 27b

In [2]:
recovery_registry_path = (
    SEMISUP_REPORTS_DIR
    / "catching_negative_repository_recovery_registry.csv"
)

negative_priority_path = (
    SEMISUP_REPORTS_DIR
    / "semisupervised_negative_site_recovery_priority.csv"
)

assert recovery_registry_path.exists()
assert negative_priority_path.exists()

recovery_registry = pd.read_csv(
    recovery_registry_path
)

negative_priority = pd.read_csv(
    negative_priority_path
)

print(
    "Recovery registry:",
    recovery_registry.shape
)

print(
    "Negative priority rows:",
    negative_priority.shape
)

display(
    recovery_registry
)


Recovery registry: (12, 10)
Negative priority rows: (16, 5)


,site_or_instance,site_key,best_status,best_status_rank,best_repository,best_repository_url,best_artifact_path,scientific_join_ready,requires_additional_validation,negative_targets
0,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,electronicsecommerceshopwithadmindashboardnextjsnodejsmain,UNKNOWN,2,Rajsu12/Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,NaN,reports/tables/catching_source_candidates.csv,False,True,H1_violation | H4_violation | H5_violation | H6_violation
1,startbootstrap-freelancer,startbootstrapfreelancer,UNKNOWN,2,StartBootstrap/startbootstrap-freelancer,NaN,reports/tables/catching_source_candidates.csv,False,True,H4_violation | H6_violation
2,startbootstrap-resume,startbootstrapresume,UNKNOWN,2,StartBootstrap/startbootstrap-resume,NaN,reports/tables/catching_source_candidates.csv,False,True,H5_violation
3,beginner-html-site-styled,beginnerhtmlsitestyled,MODERATE,4,mdn/beginner-html-site-styled,NaN,reports/tables/catching_multiple_candidate_validation.csv,False,True,H6_violation
4,charity_responsive-template,charityresponsivetemplate,UNKNOWN,2,website-templates/charity_responsive-template,NaN,reports/tables/catching_source_candidates.csv,False,True,H6_violation
5,conference-website-template,conferencewebsitetemplate,MODERATE,4,mikepierce/conference-website-template,NaN,reports/tables/catching_repository_final.csv,False,True,H6_violation
6,personal-website-template,personalwebsitetemplate,MODERATE,4,The-WebOps-Club/personal-website-template,NaN,reports/tables/catching_repository_final.csv,False,True,H6_violation
7,react-saas-template,reactsaastemplate,MODERATE,4,dunky11/react-saas-template,NaN,reports/tables/catching_repository_final.csv,False,True,H6_violation
8,simple-gallery_responsive-template,simplegalleryresponsivetemplate,UNKNOWN,2,website-templates/simple-gallery_responsive-template,NaN,reports/tables/catching_source_candidates.csv,False,True,H6_violation
9,single-page-small-business-web-application,singlepagesmallbusinesswebapplication,UNKNOWN,2,reza-azimifar/single-page-small-business-web-application,NaN,reports/tables/catching_source_candidates.csv,False,True,H6_violation


## 3. Seleccionar candidatos con identidad de repositorio

Se consideran todos los sitios negativos que tengan un `best_repository` real.

No se limita el análisis a los cuatro `MODERATE`, porque H1/H4/H5 actualmente dependen de candidatos que quedaron como `UNKNOWN` en 27b.


In [3]:
def clean_repo(value):

    if pd.isna(value):
        return ""

    text = str(value).strip()

    if (
        text == ""
        or
        text.lower()
        in {
            "nan",
            "none",
            "unknown",
            "unresolved"
        }
    ):
        return ""

    return text


candidates = recovery_registry.copy()

candidates[
    "repository"
] = candidates[
    "best_repository"
].map(
    clean_repo
)

candidates = candidates[
    candidates[
        "repository"
    ]
    !=
    ""
].copy()

print(
    "Negative sites with repository candidate:",
    len(candidates)
)

display(
    candidates[
        [
            "site_or_instance",
            "negative_targets",
            "best_status",
            "repository"
        ]
    ]
)


Negative sites with repository candidate: 12


,site_or_instance,negative_targets,best_status,repository
0,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,H1_violation | H4_violation | H5_violation | H6_violation,UNKNOWN,Rajsu12/Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main
1,startbootstrap-freelancer,H4_violation | H6_violation,UNKNOWN,StartBootstrap/startbootstrap-freelancer
2,startbootstrap-resume,H5_violation,UNKNOWN,StartBootstrap/startbootstrap-resume
3,beginner-html-site-styled,H6_violation,MODERATE,mdn/beginner-html-site-styled
4,charity_responsive-template,H6_violation,UNKNOWN,website-templates/charity_responsive-template
5,conference-website-template,H6_violation,MODERATE,mikepierce/conference-website-template
6,personal-website-template,H6_violation,MODERATE,The-WebOps-Club/personal-website-template
7,react-saas-template,H6_violation,MODERATE,dunky11/react-saas-template
8,simple-gallery_responsive-template,H6_violation,UNKNOWN,website-templates/simple-gallery_responsive-template
9,single-page-small-business-web-application,H6_violation,UNKNOWN,reza-azimifar/single-page-small-business-web-application


## 4. Funciones de normalización

In [4]:
def normalize_token(value):

    value = str(value).strip().lower()

    return re.sub(
        r"[^a-z0-9]+",
        "",
        value
    )


def repo_basename(repo_full_name):

    repo_full_name = str(
        repo_full_name
    ).strip().rstrip("/")

    if "/" in repo_full_name:
        return repo_full_name.split("/")[-1]

    return repo_full_name


def safe_dir_name(value):

    value = re.sub(
        r"[^A-Za-z0-9._-]+",
        "_",
        str(value)
    )

    return value.strip("_")


## 5. Coincidencia exacta de nombre

`exact_name_match=True` significa únicamente que:

```text
normalize(site) == normalize(repo_basename)
```

Es una evidencia de identidad, no una demostración por sí sola.


In [5]:
candidates[
    "repo_basename"
] = candidates[
    "repository"
].map(
    repo_basename
)

candidates[
    "exact_name_match"
] = [
    normalize_token(site)
    ==
    normalize_token(repo)
    for site, repo
    in zip(
        candidates[
            "site_or_instance"
        ],
        candidates[
            "repo_basename"
        ]
    )
]

display(
    candidates[
        [
            "site_or_instance",
            "repository",
            "repo_basename",
            "exact_name_match"
        ]
    ]
)


,site_or_instance,repository,repo_basename,exact_name_match
0,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,Rajsu12/Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,True
1,startbootstrap-freelancer,StartBootstrap/startbootstrap-freelancer,startbootstrap-freelancer,True
2,startbootstrap-resume,StartBootstrap/startbootstrap-resume,startbootstrap-resume,True
3,beginner-html-site-styled,mdn/beginner-html-site-styled,beginner-html-site-styled,True
4,charity_responsive-template,website-templates/charity_responsive-template,charity_responsive-template,True
5,conference-website-template,mikepierce/conference-website-template,conference-website-template,True
6,personal-website-template,The-WebOps-Club/personal-website-template,personal-website-template,True
7,react-saas-template,dunky11/react-saas-template,react-saas-template,True
8,simple-gallery_responsive-template,website-templates/simple-gallery_responsive-template,simple-gallery_responsive-template,True
9,single-page-small-business-web-application,reza-azimifar/single-page-small-business-web-application,single-page-small-business-web-application,True


## 6. Recuperar evidencia de validación anterior

Se inspeccionan:

- `catching_single_candidate_validation.csv`
- `catching_multiple_candidate_validation.csv`

y se conserva la mejor evidencia disponible para la pareja exacta `(site, repository)`.

No se hace unión por índice.


In [6]:
validation_paths = [
    REPORTS_DIR
    / "catching_single_candidate_validation.csv",

    REPORTS_DIR
    / "catching_multiple_candidate_validation.csv"
]

validation_frames = []

for path in validation_paths:

    if not path.exists():
        continue

    df = pd.read_csv(path)

    needed = {
        "site",
        "repo_full_name"
    }

    if not needed.issubset(
        df.columns
    ):
        continue

    df = df.copy()

    df[
        "validation_source_file"
    ] = str(
        path.relative_to(
            PROJECT_DIR
        )
    )

    validation_frames.append(
        df
    )

if validation_frames:

    prior_validation = pd.concat(
        validation_frames,
        ignore_index=True
    )

else:

    prior_validation = pd.DataFrame()

print(
    "Prior validation rows:",
    len(prior_validation)
)

display(
    prior_validation.head(20)
)


Prior validation rows: 73


,site,repo_full_name,clone_ok,source_files,snippets_tested,matched_snippets,matched_evaluations,matched_files,error,validation_status,validation_source_file
0,academic-website-template,sbryngelson/academic-website-template,True,31,8,0,0,NaN,NaN,NO_MATCH,reports/tables/catching_single_candidate_validation.csv
1,conference-website-template,mikepierce/conference-website-template,True,8,2,2,1,index.html,NaN,MODERATE_MATCH,reports/tables/catching_single_candidate_validation.csv
2,jekyll-inclusion,website-templates/jekyll-inclusion,True,47,7,0,0,NaN,NaN,NO_MATCH,reports/tables/catching_single_candidate_validation.csv
3,marketing-agency_one-page-template,website-templates/marketing-agency_one-page-template,True,34,1,0,0,NaN,NaN,NO_MATCH,reports/tables/catching_single_candidate_validation.csv
4,material-dashboard,laravel-frontend-presets/material-dashboard,True,156,8,0,0,NaN,NaN,NO_MATCH,reports/tables/catching_single_candidate_validation.csv
5,portfolio_one-page-template,website-templates/portfolio_one-page-template,True,34,4,0,0,NaN,NaN,NO_MATCH,reports/tables/catching_single_candidate_validation.csv
6,react-saas-template,dunky11/react-saas-template,True,99,2,1,1,src/logged_in/components/dashboard/AccountInformationArea.js,NaN,MODERATE_MATCH,reports/tables/catching_single_candidate_validation.csv
7,simple-product_responsive-template,website-templates/simple-product_responsive-template,True,21,5,0,0,NaN,NaN,NO_MATCH,reports/tables/catching_single_candidate_validation.csv
8,startbootstrap-3-col-portfolio,StartBootstrap/startbootstrap-3-col-portfolio,True,4,12,10,2,index.html,NaN,STRONG_MATCH,reports/tables/catching_single_candidate_validation.csv
9,startbootstrap-simple-sidebar,StartBootstrap/startbootstrap-simple-sidebar,True,14,11,1,1,src/js/scripts.js,NaN,MODERATE_MATCH,reports/tables/catching_single_candidate_validation.csv


In [7]:
def status_rank(value):

    text = str(
        value
    ).upper()

    if "CONFIRMED_STRONG" in text:
        return 5

    if (
        "MODERATE_MATCH" in text
        or
        "PROVISIONAL_MODERATE" in text
    ):
        return 4

    if "AMBIGUOUS" in text:
        return 3

    if (
        text == ""
        or
        text == "NAN"
        or
        "UNKNOWN" in text
    ):
        return 2

    return 1


prior_records = []

if not prior_validation.empty:

    prior_validation[
        "site_key"
    ] = prior_validation[
        "site"
    ].map(
        normalize_token
    )

    prior_validation[
        "repo_key"
    ] = prior_validation[
        "repo_full_name"
    ].map(
        lambda x:
            str(x).strip().lower()
    )

    if (
        "validation_status"
        in prior_validation.columns
    ):

        prior_validation[
            "__status_rank"
        ] = prior_validation[
            "validation_status"
        ].map(
            status_rank
        )

    else:

        prior_validation[
            "__status_rank"
        ] = 0

    for _, candidate in (
        candidates.iterrows()
    ):

        site_key = normalize_token(
            candidate[
                "site_or_instance"
            ]
        )

        repo_key = str(
            candidate[
                "repository"
            ]
        ).strip().lower()

        subset = prior_validation[
            (
                prior_validation[
                    "site_key"
                ]
                ==
                site_key
            )
            &
            (
                prior_validation[
                    "repo_key"
                ]
                ==
                repo_key
            )
        ].copy()

        if subset.empty:

            prior_records.append({
                "site_or_instance":
                    candidate[
                        "site_or_instance"
                    ],

                "repository":
                    candidate[
                        "repository"
                    ],

                "prior_validation_status":
                    "",

                "prior_clone_ok":
                    np.nan,

                "prior_source_files":
                    np.nan,

                "prior_snippets_tested":
                    np.nan,

                "prior_matched_snippets":
                    np.nan,

                "prior_matched_evaluations":
                    np.nan,

                "prior_matched_files":
                    np.nan,

                "prior_validation_source":
                    ""
            })

            continue

        subset = subset.sort_values(
            "__status_rank",
            ascending=False
        )

        best = subset.iloc[0]

        prior_records.append({
            "site_or_instance":
                candidate[
                    "site_or_instance"
                ],

            "repository":
                candidate[
                    "repository"
                ],

            "prior_validation_status":
                best.get(
                    "validation_status",
                    ""
                ),

            "prior_clone_ok":
                best.get(
                    "clone_ok",
                    np.nan
                ),

            "prior_source_files":
                best.get(
                    "source_files",
                    np.nan
                ),

            "prior_snippets_tested":
                best.get(
                    "snippets_tested",
                    np.nan
                ),

            "prior_matched_snippets":
                best.get(
                    "matched_snippets",
                    np.nan
                ),

            "prior_matched_evaluations":
                best.get(
                    "matched_evaluations",
                    np.nan
                ),

            "prior_matched_files":
                best.get(
                    "matched_files",
                    np.nan
                ),

            "prior_validation_source":
                best.get(
                    "validation_source_file",
                    ""
                )
        })

prior_evidence = pd.DataFrame(
    prior_records
)

candidates = candidates.merge(
    prior_evidence,
    on=[
        "site_or_instance",
        "repository"
    ],
    how="left",
    validate="one_to_one"
)

display(
    candidates[
        [
            "site_or_instance",
            "repository",
            "best_status",
            "prior_validation_status",
            "prior_matched_snippets",
            "prior_matched_evaluations"
        ]
    ]
)


,site_or_instance,repository,best_status,prior_validation_status,prior_matched_snippets,prior_matched_evaluations
0,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,Rajsu12/Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,UNKNOWN,NO_TESTABLE_EVIDENCE,0,0
1,startbootstrap-freelancer,StartBootstrap/startbootstrap-freelancer,UNKNOWN,NO_MATCH,0,0
2,startbootstrap-resume,StartBootstrap/startbootstrap-resume,UNKNOWN,NO_MATCH,0,0
3,beginner-html-site-styled,mdn/beginner-html-site-styled,MODERATE,MODERATE_MATCH,1,1
4,charity_responsive-template,website-templates/charity_responsive-template,UNKNOWN,NO_MATCH,0,0
5,conference-website-template,mikepierce/conference-website-template,MODERATE,MODERATE_MATCH,2,1
6,personal-website-template,The-WebOps-Club/personal-website-template,MODERATE,MODERATE_MATCH,2,1
7,react-saas-template,dunky11/react-saas-template,MODERATE,MODERATE_MATCH,1,1
8,simple-gallery_responsive-template,website-templates/simple-gallery_responsive-template,UNKNOWN,NO_MATCH,0,0
9,single-page-small-business-web-application,reza-azimifar/single-page-small-business-web-application,UNKNOWN,NO_TESTABLE_EVIDENCE,0,0


## 7. Verificación remota y clone superficial

Este paso requiere conexión a Internet en la máquina donde se ejecute el notebook.

Se usa:

```bash
git ls-remote --heads
git clone --depth 1
```

La carpeta de clone es nueva y específica para esta fase.

Si ya existe un clone Git válido, se reutiliza. No se elimina ni sobrescribe.


In [8]:
def run_command(
    command,
    timeout=60
):

    try:

        result = subprocess.run(
            command,
            capture_output=True,
            text=True,
            timeout=timeout
        )

        return {
            "returncode":
                result.returncode,

            "stdout":
                result.stdout,

            "stderr":
                result.stderr,

            "error":
                ""
        }

    except Exception as exc:

        return {
            "returncode":
                -1,

            "stdout":
                "",

            "stderr":
                "",

            "error":
                repr(exc)
        }


technical_records = []

for _, row in (
    candidates.iterrows()
):

    site = row[
        "site_or_instance"
    ]

    repo = row[
        "repository"
    ]

    site_key = normalize_token(
        site
    )

    remote_url = (
        f"https://github.com/{repo}.git"
    )

    local_dir = (
        RECOVERY_ROOT
        /
        (
            safe_dir_name(
                site_key
            )
            +
            "__"
            +
            safe_dir_name(
                repo.replace(
                    "/",
                    "__"
                )
            )
        )
    )

    ls_remote = run_command(
        [
            "git",
            "ls-remote",
            "--heads",
            remote_url
        ],
        timeout=45
    )

    remote_exists = (
        ls_remote[
            "returncode"
        ]
        ==
        0
        and
        bool(
            ls_remote[
                "stdout"
            ].strip()
        )
    )

    reused_existing_clone = False
    clone_ok = False
    clone_error = ""

    if (
        local_dir.exists()
        and
        (
            local_dir
            /
            ".git"
        ).exists()
    ):

        reused_existing_clone = True
        clone_ok = True

    elif local_dir.exists():

        clone_error = (
            "Destination exists but is not "
            "a Git repository; left untouched."
        )

    elif remote_exists:

        clone_result = run_command(
            [
                "git",
                "clone",
                "--depth",
                "1",
                remote_url,
                str(
                    local_dir
                )
            ],
            timeout=180
        )

        clone_ok = (
            clone_result[
                "returncode"
            ]
            ==
            0
            and
            (
                local_dir
                /
                ".git"
            ).exists()
        )

        if not clone_ok:

            clone_error = (
                clone_result[
                    "error"
                ]
                or
                clone_result[
                    "stderr"
                ]
            )

    else:

        clone_error = (
            ls_remote[
                "error"
            ]
            or
            ls_remote[
                "stderr"
            ]
        )

    web_source_files = []

    if clone_ok:

        for path in local_dir.rglob("*"):

            if not path.is_file():
                continue

            if ".git" in path.parts:
                continue

            if (
                path.suffix.lower()
                in
                WEB_EXTENSIONS
            ):

                web_source_files.append(
                    path
                )

    technical_records.append({
        "site_or_instance":
            site,

        "repository":
            repo,

        "remote_url":
            remote_url,

        "remote_exists":
            remote_exists,

        "local_path":
            str(
                local_dir.relative_to(
                    PROJECT_DIR
                )
            ),

        "reused_existing_clone":
            reused_existing_clone,

        "clone_ok":
            clone_ok,

        "web_source_file_count":
            len(
                web_source_files
            ),

        "has_web_source":
            len(
                web_source_files
            )
            >
            0,

        "clone_error":
            clone_error[:1000]
    })

technical_validation = pd.DataFrame(
    technical_records
)

display(
    technical_validation
)


,site_or_instance,repository,remote_url,remote_exists,local_path,reused_existing_clone,clone_ok,web_source_file_count,has_web_source,clone_error
0,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,Rajsu12/Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,https://github.com/Rajsu12/Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main.git,True,data/public_v2/source_recovery/catching_negative_candidates/electronicsecommerceshopwithadmindashboardnextjsnodejsmain__Rajsu12__Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,False,True,4,True,
1,startbootstrap-freelancer,StartBootstrap/startbootstrap-freelancer,https://github.com/StartBootstrap/startbootstrap-freelancer.git,True,data/public_v2/source_recovery/catching_negative_candidates/startbootstrapfreelancer__StartBootstrap__startbootstrap-freelancer,False,True,31,True,
2,startbootstrap-resume,StartBootstrap/startbootstrap-resume,https://github.com/StartBootstrap/startbootstrap-resume.git,True,data/public_v2/source_recovery/catching_negative_candidates/startbootstrapresume__StartBootstrap__startbootstrap-resume,False,True,25,True,
3,beginner-html-site-styled,mdn/beginner-html-site-styled,https://github.com/mdn/beginner-html-site-styled.git,True,data/public_v2/source_recovery/catching_negative_candidates/beginnerhtmlsitestyled__mdn__beginner-html-site-styled,False,True,2,True,
4,charity_responsive-template,website-templates/charity_responsive-template,https://github.com/website-templates/charity_responsive-template.git,True,data/public_v2/source_recovery/catching_negative_candidates/charityresponsivetemplate__website-templates__charity_responsive-template,False,True,22,True,
5,conference-website-template,mikepierce/conference-website-template,https://github.com/mikepierce/conference-website-template.git,True,data/public_v2/source_recovery/catching_negative_candidates/conferencewebsitetemplate__mikepierce__conference-website-template,False,True,6,True,
6,personal-website-template,The-WebOps-Club/personal-website-template,https://github.com/The-WebOps-Club/personal-website-template.git,True,data/public_v2/source_recovery/catching_negative_candidates/personalwebsitetemplate__The-WebOps-Club__personal-website-template,False,True,43,True,
7,react-saas-template,dunky11/react-saas-template,https://github.com/dunky11/react-saas-template.git,True,data/public_v2/source_recovery/catching_negative_candidates/reactsaastemplate__dunky11__react-saas-template,False,True,98,True,
8,simple-gallery_responsive-template,website-templates/simple-gallery_responsive-template,https://github.com/website-templates/simple-gallery_responsive-template.git,True,data/public_v2/source_recovery/catching_negative_candidates/simplegalleryresponsivetemplate__website-templates__simple-gallery_responsive-template,False,True,30,True,
9,single-page-small-business-web-application,reza-azimifar/single-page-small-business-web-application,https://github.com/reza-azimifar/single-page-small-business-web-application.git,True,data/public_v2/source_recovery/catching_negative_candidates/singlepagesmallbusinesswebapplication__reza-azimifar__single-page-small-business-web-application,False,True,25,True,


## 8. Integrar evidencia y asignar tier

Reglas operacionales:

### A — `A_CONFIRMED_STRONG`

Requiere evidencia previa `CONFIRMED_STRONG` y acceso técnico correcto.

### B — `B_MODERATE_VALIDATED`

Requiere evidencia previa `MODERATE_MATCH`/`PROVISIONAL_MODERATE` y acceso técnico correcto.

### C — `C_EXACT_NAME_TECHNICAL`

No tiene evidencia A/B, pero:

- el nombre normalizado del repositorio coincide exactamente con el sitio;
- el repositorio existe;
- se pudo clonar;
- contiene fuente web.

### D — `D_ACCESSIBLE_CANDIDATE`

Repositorio accesible con fuente web, pero sin evidencia suficiente de identidad.

### E — `E_NOT_RECOVERABLE`

No se logró recuperar técnicamente.

B y C **no son equivalentes a A**.


In [9]:
validated = candidates.merge(
    technical_validation,
    on=[
        "site_or_instance",
        "repository"
    ],
    how="left",
    validate="one_to_one"
)


def assign_evidence_tier(row):

    status_text = (
        str(
            row.get(
                "prior_validation_status",
                ""
            )
        )
        +
        " "
        +
        str(
            row.get(
                "best_status",
                ""
            )
        )
    ).upper()

    technical_ok = bool(
        row.get(
            "remote_exists",
            False
        )
        and
        row.get(
            "clone_ok",
            False
        )
        and
        row.get(
            "has_web_source",
            False
        )
    )

    if (
        technical_ok
        and
        "CONFIRMED_STRONG"
        in
        status_text
    ):

        return (
            "A_CONFIRMED_STRONG"
        )

    if (
        technical_ok
        and
        (
            "MODERATE_MATCH"
            in
            status_text
            or
            "MODERATE"
            in
            status_text
        )
    ):

        return (
            "B_MODERATE_VALIDATED"
        )

    if (
        technical_ok
        and
        bool(
            row.get(
                "exact_name_match",
                False
            )
        )
    ):

        return (
            "C_EXACT_NAME_TECHNICAL"
        )

    if technical_ok:

        return (
            "D_ACCESSIBLE_CANDIDATE"
        )

    return (
        "E_NOT_RECOVERABLE"
    )


validated[
    "evidence_tier"
] = validated.apply(
    assign_evidence_tier,
    axis=1
)

validated[
    "strict_source_match"
] = (
    validated[
        "evidence_tier"
    ]
    ==
    "A_CONFIRMED_STRONG"
)

validated[
    "extended_source_match"
] = validated[
    "evidence_tier"
].isin(
    [
        "A_CONFIRMED_STRONG",
        "B_MODERATE_VALIDATED",
        "C_EXACT_NAME_TECHNICAL"
    ]
)

validated[
    "eligible_for_feature_extraction"
] = validated[
    "extended_source_match"
]

display(
    validated[
        [
            "site_or_instance",
            "negative_targets",
            "repository",
            "exact_name_match",
            "prior_validation_status",
            "remote_exists",
            "clone_ok",
            "web_source_file_count",
            "evidence_tier",
            "strict_source_match",
            "extended_source_match"
        ]
    ]
)


,site_or_instance,negative_targets,repository,exact_name_match,prior_validation_status,remote_exists,clone_ok,web_source_file_count,evidence_tier,strict_source_match,extended_source_match
0,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,H1_violation | H4_violation | H5_violation | H6_violation,Rajsu12/Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,True,NO_TESTABLE_EVIDENCE,True,True,4,C_EXACT_NAME_TECHNICAL,False,True
1,startbootstrap-freelancer,H4_violation | H6_violation,StartBootstrap/startbootstrap-freelancer,True,NO_MATCH,True,True,31,C_EXACT_NAME_TECHNICAL,False,True
2,startbootstrap-resume,H5_violation,StartBootstrap/startbootstrap-resume,True,NO_MATCH,True,True,25,C_EXACT_NAME_TECHNICAL,False,True
3,beginner-html-site-styled,H6_violation,mdn/beginner-html-site-styled,True,MODERATE_MATCH,True,True,2,B_MODERATE_VALIDATED,False,True
4,charity_responsive-template,H6_violation,website-templates/charity_responsive-template,True,NO_MATCH,True,True,22,C_EXACT_NAME_TECHNICAL,False,True
5,conference-website-template,H6_violation,mikepierce/conference-website-template,True,MODERATE_MATCH,True,True,6,B_MODERATE_VALIDATED,False,True
6,personal-website-template,H6_violation,The-WebOps-Club/personal-website-template,True,MODERATE_MATCH,True,True,43,B_MODERATE_VALIDATED,False,True
7,react-saas-template,H6_violation,dunky11/react-saas-template,True,MODERATE_MATCH,True,True,98,B_MODERATE_VALIDATED,False,True
8,simple-gallery_responsive-template,H6_violation,website-templates/simple-gallery_responsive-template,True,NO_MATCH,True,True,30,C_EXACT_NAME_TECHNICAL,False,True
9,single-page-small-business-web-application,H6_violation,reza-azimifar/single-page-small-business-web-application,True,NO_TESTABLE_EVIDENCE,True,True,25,C_EXACT_NAME_TECHNICAL,False,True


## 9. Prioridad metodológica

Se priorizan primero los sitios que aportan negativos a H1/H4/H5, porque esas heurísticas actualmente no tienen ningún candidato `MODERATE` en 27b.

Después se consideran sitios exclusivos de H6.


In [10]:
def priority_level(negative_targets):

    text = str(
        negative_targets
    )

    if any(
        target in text
        for target in [
            "H1_violation",
            "H4_violation",
            "H5_violation"
        ]
    ):

        return 1

    if "H6_violation" in text:

        return 2

    return 3


validated[
    "recovery_priority"
] = validated[
    "negative_targets"
].map(
    priority_level
)

priority_registry = validated.sort_values(
    [
        "recovery_priority",
        "extended_source_match",
        "evidence_tier"
    ],
    ascending=[
        True,
        False,
        True
    ]
).reset_index(
    drop=True
)

display(
    priority_registry[
        [
            "recovery_priority",
            "site_or_instance",
            "negative_targets",
            "repository",
            "evidence_tier",
            "eligible_for_feature_extraction"
        ]
    ]
)


,recovery_priority,site_or_instance,negative_targets,repository,evidence_tier,eligible_for_feature_extraction
0,1,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,H1_violation | H4_violation | H5_violation | H6_violation,Rajsu12/Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,C_EXACT_NAME_TECHNICAL,True
1,1,startbootstrap-freelancer,H4_violation | H6_violation,StartBootstrap/startbootstrap-freelancer,C_EXACT_NAME_TECHNICAL,True
2,1,startbootstrap-resume,H5_violation,StartBootstrap/startbootstrap-resume,C_EXACT_NAME_TECHNICAL,True
3,2,beginner-html-site-styled,H6_violation,mdn/beginner-html-site-styled,B_MODERATE_VALIDATED,True
4,2,conference-website-template,H6_violation,mikepierce/conference-website-template,B_MODERATE_VALIDATED,True
5,2,personal-website-template,H6_violation,The-WebOps-Club/personal-website-template,B_MODERATE_VALIDATED,True
6,2,react-saas-template,H6_violation,dunky11/react-saas-template,B_MODERATE_VALIDATED,True
7,2,charity_responsive-template,H6_violation,website-templates/charity_responsive-template,C_EXACT_NAME_TECHNICAL,True
8,2,simple-gallery_responsive-template,H6_violation,website-templates/simple-gallery_responsive-template,C_EXACT_NAME_TECHNICAL,True
9,2,single-page-small-business-web-application,H6_violation,reza-azimifar/single-page-small-business-web-application,C_EXACT_NAME_TECHNICAL,True


## 10. Cobertura de negativos recuperables por target

In [11]:
coverage_records = []

for target in TARGETS:

    subset = validated[
        validated[
            "negative_targets"
        ]
        .fillna("")
        .astype(str)
        .str.contains(
            target,
            regex=False
        )
    ].copy()

    strict_count = int(
        subset[
            "strict_source_match"
        ].sum()
    )

    extended_count = int(
        subset[
            "extended_source_match"
        ].sum()
    )

    coverage_records.append({
        "target":
            target,

        "negative_sites_known":
            len(
                subset
            ),

        "strict_negative_sources_ready":
            strict_count,

        "extended_negative_sources_ready":
            extended_count,

        "strict_negative_class_recoverable":
            strict_count > 0,

        "extended_negative_class_recoverable":
            extended_count > 0
    })

target_negative_coverage = pd.DataFrame(
    coverage_records
)

display(
    target_negative_coverage
)


,target,negative_sites_known,strict_negative_sources_ready,extended_negative_sources_ready,strict_negative_class_recoverable,extended_negative_class_recoverable
0,H1_violation,1,0,1,False,True
1,H4_violation,2,0,2,False,True
2,H5_violation,2,0,2,False,True
3,H6_violation,11,0,11,False,True


## 11. Decisión global

La extracción de X puede continuar en modo extendido únicamente si **cada target** tiene al menos un sitio negativo con tier A/B/C.

Esto no convierte B/C en fuente estricta: esa diferencia se conservará en el dataset final mediante `source_match_level`.


In [12]:
strict_all_targets = bool(
    target_negative_coverage[
        "strict_negative_class_recoverable"
    ].all()
)

extended_all_targets = bool(
    target_negative_coverage[
        "extended_negative_class_recoverable"
    ].all()
)

extended_any_target = bool(
    target_negative_coverage[
        "extended_negative_class_recoverable"
    ].any()
)

if strict_all_targets:

    global_status = (
        "STRICT_NEGATIVE_X_EXTRACTION_CAN_PROCEED"
    )

    next_action = (
        "Extract proxy X from tier A repositories."
    )

elif extended_all_targets:

    global_status = (
        "EXTENDED_NEGATIVE_X_EXTRACTION_CAN_PROCEED"
    )

    next_action = (
        "Extract proxy X from A/B/C repositories "
        "while preserving source-match tier."
    )

elif extended_any_target:

    global_status = (
        "PARTIAL_NEGATIVE_X_EXTRACTION_CAN_PROCEED"
    )

    next_action = (
        "Only some heuristics have recoverable "
        "negative source evidence."
    )

else:

    global_status = (
        "NO_NEGATIVE_X_EXTRACTION_PATH"
    )

    next_action = (
        "Do not proceed to baseline; current "
        "candidate validation is insufficient."
    )

technical_validation_decision = pd.DataFrame([
    {
        "global_status":
            global_status,

        "strict_all_targets":
            strict_all_targets,

        "extended_all_targets":
            extended_all_targets,

        "eligible_repositories":
            int(
                validated[
                    "eligible_for_feature_extraction"
                ].sum()
            ),

        "next_action":
            next_action
    }
])

display(
    technical_validation_decision
)


,global_status,strict_all_targets,extended_all_targets,eligible_repositories,next_action
0,EXTENDED_NEGATIVE_X_EXTRACTION_CAN_PROCEED,False,True,12,Extract proxy X from A/B/C repositories while preserving source-match tier.


## 12. Tabla para la futura extracción de X

Solo se exportan a la cola de extracción los tiers A/B/C.

La etiqueta Nielsen no se modifica. La extracción posterior debe unir por `site_or_instance`, nunca por índice.


In [13]:
feature_extraction_queue = validated[
    validated[
        "eligible_for_feature_extraction"
    ]
].copy()

queue_columns = [
    "site_or_instance",
    "negative_targets",
    "repository",
    "local_path",
    "evidence_tier",
    "strict_source_match",
    "extended_source_match",
    "prior_validation_status",
    "exact_name_match",
    "web_source_file_count"
]

feature_extraction_queue = feature_extraction_queue[
    queue_columns
].sort_values(
    [
        "strict_source_match",
        "evidence_tier",
        "site_or_instance"
    ],
    ascending=[
        False,
        True,
        True
    ]
).reset_index(
    drop=True
)

display(
    feature_extraction_queue
)


,site_or_instance,negative_targets,repository,local_path,evidence_tier,strict_source_match,extended_source_match,prior_validation_status,exact_name_match,web_source_file_count
0,beginner-html-site-styled,H6_violation,mdn/beginner-html-site-styled,data/public_v2/source_recovery/catching_negative_candidates/beginnerhtmlsitestyled__mdn__beginner-html-site-styled,B_MODERATE_VALIDATED,False,True,MODERATE_MATCH,True,2
1,conference-website-template,H6_violation,mikepierce/conference-website-template,data/public_v2/source_recovery/catching_negative_candidates/conferencewebsitetemplate__mikepierce__conference-website-template,B_MODERATE_VALIDATED,False,True,MODERATE_MATCH,True,6
2,personal-website-template,H6_violation,The-WebOps-Club/personal-website-template,data/public_v2/source_recovery/catching_negative_candidates/personalwebsitetemplate__The-WebOps-Club__personal-website-template,B_MODERATE_VALIDATED,False,True,MODERATE_MATCH,True,43
3,react-saas-template,H6_violation,dunky11/react-saas-template,data/public_v2/source_recovery/catching_negative_candidates/reactsaastemplate__dunky11__react-saas-template,B_MODERATE_VALIDATED,False,True,MODERATE_MATCH,True,98
4,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,H1_violation | H4_violation | H5_violation | H6_violation,Rajsu12/Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,data/public_v2/source_recovery/catching_negative_candidates/electronicsecommerceshopwithadmindashboardnextjsnodejsmain__Rajsu12__Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,C_EXACT_NAME_TECHNICAL,False,True,NO_TESTABLE_EVIDENCE,True,4
5,charity_responsive-template,H6_violation,website-templates/charity_responsive-template,data/public_v2/source_recovery/catching_negative_candidates/charityresponsivetemplate__website-templates__charity_responsive-template,C_EXACT_NAME_TECHNICAL,False,True,NO_MATCH,True,22
6,simple-gallery_responsive-template,H6_violation,website-templates/simple-gallery_responsive-template,data/public_v2/source_recovery/catching_negative_candidates/simplegalleryresponsivetemplate__website-templates__simple-gallery_responsive-template,C_EXACT_NAME_TECHNICAL,False,True,NO_MATCH,True,30
7,single-page-small-business-web-application,H6_violation,reza-azimifar/single-page-small-business-web-application,data/public_v2/source_recovery/catching_negative_candidates/singlepagesmallbusinesswebapplication__reza-azimifar__single-page-small-business-web-application,C_EXACT_NAME_TECHNICAL,False,True,NO_TESTABLE_EVIDENCE,True,25
8,startbootstrap-freelancer,H4_violation | H6_violation,StartBootstrap/startbootstrap-freelancer,data/public_v2/source_recovery/catching_negative_candidates/startbootstrapfreelancer__StartBootstrap__startbootstrap-freelancer,C_EXACT_NAME_TECHNICAL,False,True,NO_MATCH,True,31
9,startbootstrap-landing-page,H6_violation,StartBootstrap/startbootstrap-landing-page,data/public_v2/source_recovery/catching_negative_candidates/startbootstraplandingpage__StartBootstrap__startbootstrap-landing-page,C_EXACT_NAME_TECHNICAL,False,True,NO_MATCH,True,28


## 13. Guardar resultados

In [14]:
validated.to_csv(
    SEMISUP_REPORTS_DIR
    / "catching_negative_candidate_technical_validation.csv",
    index=False
)

priority_registry.to_csv(
    SEMISUP_REPORTS_DIR
    / "catching_negative_candidate_priority_registry.csv",
    index=False
)

target_negative_coverage.to_csv(
    SEMISUP_REPORTS_DIR
    / "catching_negative_candidate_coverage_by_target.csv",
    index=False
)

technical_validation_decision.to_csv(
    SEMISUP_REPORTS_DIR
    / "catching_negative_candidate_technical_decision.csv",
    index=False
)

feature_extraction_queue.to_csv(
    SEMISUP_REPORTS_DIR
    / "catching_negative_feature_extraction_queue.csv",
    index=False
)


In [15]:
expected_outputs = [
    "catching_negative_candidate_technical_validation.csv",
    "catching_negative_candidate_priority_registry.csv",
    "catching_negative_candidate_coverage_by_target.csv",
    "catching_negative_candidate_technical_decision.csv",
    "catching_negative_feature_extraction_queue.csv"
]

for filename in expected_outputs:

    path = (
        SEMISUP_REPORTS_DIR
        /
        filename
    )

    print(
        filename,
        "✅"
        if path.exists()
        else "❌"
    )


catching_negative_candidate_technical_validation.csv ✅
catching_negative_candidate_priority_registry.csv ✅
catching_negative_candidate_coverage_by_target.csv ✅
catching_negative_candidate_technical_decision.csv ✅
catching_negative_feature_extraction_queue.csv ✅


## Punto de detención

No ejecutar todavía `28_supervised_baseline.ipynb`.

Si el resultado es:

```text
EXTENDED_NEGATIVE_X_EXTRACTION_CAN_PROCEED
```

el siguiente notebook deberá extraer exactamente:

```text
structure_depth
interactive_control_count
interactive_label_ratio
```

de los repositorios A/B/C y reconstruir el dataset maestro, conservando:

```text
source_match_level
A / B / C
```

Si el resultado es parcial, solo podrán continuar los targets que tengan soporte negativo recuperable.

Si no existe ruta de extracción, no se crearán negativos sintéticos ni se convertirán datos `U` en negativos.
